# Havnsø – Probabilistic Pressure and Injectivity Screening

This is the second decision gate after the Havnsø static-capacity notebook. In every Monte Carlo iteration it checks:

1. Is sampled static capacity at least the project target?
2. Can each well support the selected injection rate?
3. Does the screening pressure remain at or below the pressure endpoint?

An iteration succeeds only when **all three criteria pass together**.

In [ ]:
#@title Install dependencies { display-mode: "form" }
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
#@title Import libraries { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import (
    Distribution, StorageSite, TechnicalScreeningCase,
    simulate, simulate_technical_screening,
)
plt.style.use("seaborn-v0_8-whitegrid")

## What is reported and what is derived

The pressure/injectivity calculation is a **GEUS-normalized screening surrogate**, not an Eclipse 100 reproduction. GEUS Report 2020/48 publishes a reference case and qualitative sensitivities, but no transferable pressure equation, numerical outcome for every sensitivity, or probability weights.

The surrogate exactly reaches the reference endpoint when all inputs equal the 2020 base case. It scales pressure demand with cumulative mass, total field rate, inverse permeability factor, and inverse N/G. It scales the per-well injectivity limit with permeability factor and N/G.

The updated 2023 static-capacity samples remain the current capacity evidence. The older version-0 dynamic result is used only to normalize this provisional technical gate.

## Editable inputs

The default **60 Mt** target is selected near the updated GEUS 2023/38 P50 of 62.82 Mt. It is an editable screening target, not a GEUS development plan.

In [ ]:
# Updated static-capacity inputs: GEUS 2023/38, Scenario 1.
grv_km3 = (2.9, 5.0, 8.0)
net_to_gross = (0.60, 0.75, 0.90)
porosity = (0.175, 0.219, 0.263)
co2_density_kg_m3 = (663.86, 698.8, 768.68)
storage_efficiency = (0.05, 0.10, 0.20)

# User-editable project controls.
target_mass_mt = 60.0 #@param {type:"number"}
number_of_wells = 3 #@param {type:"integer"}
rate_mtpy_per_well = 1.0 #@param {type:"number"}
iterations = 100000 #@param {type:"integer"}
random_seed = 42 #@param {type:"integer"}

# GEUS 2020/48 reference case and sensitivities.
initial_pressure_bar = 130.0
pressure_endpoint_bar = 240.0
reference_mass_mt = 270.0
reference_wells = 3
reference_rate_mtpy_per_well = 1.0
reference_net_to_gross = 0.5
permeability_factor = (0.5, 1.0, 2.0)

In [ ]:
#@title Show input parameter table { display-mode: "form" }
input_table = pd.DataFrame([
    ["Gross rock volume", "GRV", "km³", "PERT", *grv_km3, "Capacity", "Seismic-derived estimate", "GEUS 2023/38, Table 8.2.3"],
    ["Net-to-gross", "N/G", "fraction", "PERT", *net_to_gross, "Capacity, pressure, injectivity", "2023 analogue prognosis; sampled value is used by the surrogate", "GEUS 2023/38, Table 8.2.3"],
    ["Porosity", "φ", "fraction", "PERT", *porosity, "Capacity; poro-perm illustration", "Stenlille-based 2023 prognosis; not a Havnsø measurement", "GEUS 2023/38, Table 8.2.3"],
    ["In-situ CO₂ density", "ρCO₂", "kg/m³", "PERT", *co2_density_kg_m3, "Capacity", "Thermodynamic estimate", "GEUS 2023/38, Table 8.2.3"],
    ["Storage efficiency", "Seff", "fraction", "PERT", *storage_efficiency, "Capacity", "Screening assumption", "GEUS 2023/38, Table 8.2.3"],
    ["Permeability multiplier", "kfactor", "multiplier", "PERT", *permeability_factor, "Pressure and injectivity", "GEUS scenarios converted here to a probability model", "GEUS 2020/48, Figures 7-8"],
    ["Initial datum pressure", "P0", "bar", "Fixed", None, initial_pressure_bar, None, "Pressure", "GEUS reported", "GEUS 2020/48, Table 1"],
    ["Pressure endpoint", "Plim", "bar absolute", "Fixed", None, pressure_endpoint_bar, None, "Pressure criterion", "Interpretation of ambiguous GEUS wording", "GEUS 2020/48; note below"],
    ["Reference injected mass", "Mref", "Mt", "Fixed", None, reference_mass_mt, None, "Pressure normalization", "GEUS reported", "GEUS 2020/48, base case"],
    ["Reference wells", "nref", "count", "Fixed", None, reference_wells, None, "Pressure normalization", "GEUS reported", "GEUS 2020/48, base case"],
    ["Reference rate per well", "qref", "Mt/year/well", "Fixed", None, reference_rate_mtpy_per_well, None, "Pressure and injectivity normalization", "GEUS reported", "GEUS 2020/48, base case"],
    ["Reference N/G", "(N/G)ref", "fraction", "Fixed", None, reference_net_to_gross, None, "Pressure and injectivity normalization", "GEUS reported", "GEUS 2020/48, base case"],
    ["Project target", "Mtarget", "Mt", "Editable", None, target_mass_mt, None, "All criteria", "User-selected assumption", "Default near GEUS 2023/38 P50"],
    ["Project wells", "n", "count", "Editable", None, number_of_wells, None, "Duration and pressure", "User-selected assumption", "Colab control"],
    ["Rate per project well", "qwell", "Mt/year/well", "Editable", None, rate_mtpy_per_well, None, "Duration, pressure and injectivity", "User-selected assumption", "Colab control"],
], columns=["Parameter", "Symbol", "Unit", "Distribution / value", "Minimum", "Mode / reference", "Maximum", "Used in", "Evidence status", "Source / basis"])
input_table

**Pressure interpretation:** the English summary calls 240 bar an “overpressure”. This notebook treats 240 bar as the absolute endpoint pressure. Adding 240 bar to the 130-bar initial pressure would conflict with the report's statement that the 75%-of-lithostatic fracture constraint was respected. Keep this interpretation visible until the original simulation files or an updated model resolve the terminology.

## Formulas used in this notebook

### Static capacity

$$SC_i=GRV_i(N/G)_i\phi_i\rho_{CO_2,i}S_{eff,i}$$

### Injection duration

$$t=\frac{M_{target}}{nq_{well}}$$

### Pressure screening surrogate

$$R_{P,i}=\frac{M_{target}}{M_{ref}}\frac{nq_{well}}{n_{ref}q_{ref}}\frac{1}{k_{factor,i}}\frac{(N/G)_{ref}}{(N/G)_i}$$

$$P_{final,i}=P_0+(P_{lim}-P_0)R_{P,i}$$

### Per-well injectivity screening

$$q_{max,i}=q_{ref}k_{factor,i}\frac{(N/G)_i}{(N/G)_{ref}}$$

### Pass criteria and probability

$$I_{success,i}=I(SC_i\ge M_{target})\land I(q_{well}\le q_{max,i})\land I(P_{final,i}\le P_{lim})$$

$$P(success)=\frac{1}{N}\sum_{i=1}^{N}I_{success,i}$$

The capacity equation follows the GEUS static assessment. The pressure and injectivity equations above are **project screening surrogates created for this notebook**; they are normalized to GEUS 2020/48 but were not published by GEUS and do not replace Eclipse reservoir simulation.

## Stenlille porosity-permeability relationships

GEUS 2020/48 Figure 1 used conventional core-analysis data from Stenlille wells to populate Gassum Formation sandstone permeability in the version-0 dynamic model. With porosity expressed in percent:

$$k_{gas,upper}=0.000031\phi_{\%}^{4.91811}$$

$$k_{gas,lower}=0.00028\phi_{\%}^{4.91811}$$

GEUS then applied $k_{fluid}=0.5k_{gas}$. Permeability is in mD. These relationships are **Stenlille analogues, not Havnsø core measurements**. The chart below evaluates them across the updated 2023 Havnsø porosity range only to make their magnitude visible. The current surrogate still uses the dimensionless $k_{factor}$ distribution rather than these absolute permeability values.

In [ ]:
#@title Show Stenlille porosity-permeability analogue { display-mode: "form" }
def stenlille_permeability(phi_fraction, coefficient):
    phi_percent = np.asarray(phi_fraction) * 100.0
    gas_permeability_md = coefficient * phi_percent ** 4.91811
    return 0.5 * gas_permeability_md

poro_perm_table = pd.DataFrame({
    "Porosity case": ["Minimum", "Mode", "Maximum"],
    "Porosity (%)": np.array(porosity) * 100.0,
    "Upper Sands fluid k (mD)": stenlille_permeability(porosity, 0.000031),
    "Lower Sands fluid k (mD)": stenlille_permeability(porosity, 0.00028),
})
display(poro_perm_table.round(2))

phi_plot = np.linspace(porosity[0], porosity[2], 200)
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(phi_plot * 100, stenlille_permeability(phi_plot, 0.000031), label="Upper Sands - fluid permeability", linewidth=2)
ax.semilogy(phi_plot * 100, stenlille_permeability(phi_plot, 0.00028), label="Lower Sands - fluid permeability", linewidth=2)
for label, phi in zip(["Minimum", "Mode", "Maximum"], porosity):
    ax.axvline(phi * 100, color="#666666", linestyle="--", alpha=0.45)
    ax.text(phi * 100, ax.get_ylim()[0] * 1.15, label, rotation=90, va="bottom", ha="right")
ax.set(xlabel="Porosity (%)", ylabel="Estimated fluid permeability (mD, logarithmic scale)", title="Stenlille Gassum Formation porosity-permeability analogues")
ax.legend()
plt.show()

In [ ]:
#@title Run static capacity Monte Carlo { display-mode: "form" }
site = StorageSite(
    name="Havnsø – Gassum Formation – Scenario 1",
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)
capacity_result = simulate(site, iterations=iterations, seed=random_seed)

In [ ]:
#@title Run pressure and injectivity screening { display-mode: "form" }
technical_case = TechnicalScreeningCase(
    name=f"Havnsø – {target_mass_mt:g} Mt technical screening",
    target_mass_mt=target_mass_mt,
    wells=number_of_wells,
    rate_mtpy_per_well=rate_mtpy_per_well,
    permeability_factor=Distribution.pert(*permeability_factor),
    initial_pressure_bar=initial_pressure_bar,
    pressure_limit_bar=pressure_endpoint_bar,
    reference_mass_mt=reference_mass_mt,
    reference_wells=reference_wells,
    reference_rate_mtpy_per_well=reference_rate_mtpy_per_well,
    reference_net_to_gross=reference_net_to_gross,
)
technical_result = simulate_technical_screening(
    technical_case, capacity_result, seed=random_seed + 1
)

In [ ]:
#@title Show integrated technical-screening result { display-mode: "form" }
summary = technical_result.summary()
result_table = pd.DataFrame([
    ["Project target", f"{target_mass_mt:.1f} Mt"],
    ["Injection duration", f"{technical_result.duration_years:.1f} years"],
    ["Capacity passes", f"{summary['capacity_pass_probability']:.1%}"],
    ["Injectivity passes", f"{summary['injectivity_pass_probability']:.1%}"],
    ["Pressure passes", f"{summary['pressure_pass_probability']:.1%}"],
    ["All criteria pass (technical success)", f"{summary['success_probability']:.1%}"],
    ["At least one criterion fails", f"{summary['failure_probability']:.1%}"],
    ["P90 final pressure", f"{summary['p90_final_pressure_bar']:.1f} bar"],
    ["P50 final pressure", f"{summary['p50_final_pressure_bar']:.1f} bar"],
], columns=["Metric", "Result"])
result_table

In [ ]:
#@title Show criterion pass probabilities { display-mode: "form" }
fig, ax = technical_result.plot_criteria()
plt.show()

In [ ]:
#@title Show capacity and injectivity decision limits { display-mode: "form" }
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(capacity_result.capacity_mt, bins=50, color="#9ecae1", edgecolor="white")
axes[0].axvline(target_mass_mt, color="#c00000", linestyle="--", linewidth=2, label=f"Target: {target_mass_mt:g} Mt")
axes[0].set(xlabel="Static capacity (Mt)", ylabel="Iterations", title="Capacity gate")
axes[0].legend()

axes[1].hist(technical_result.injectivity_limit_mtpy_per_well, bins=50, color="#a1d99b", edgecolor="white")
axes[1].axvline(rate_mtpy_per_well, color="#c00000", linestyle="--", linewidth=2, label=f"Selected rate: {rate_mtpy_per_well:g} Mt/year/well")
axes[1].set(xlabel="Screening injectivity limit (Mt/year/well)", ylabel="Iterations", title="Injectivity gate")
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
#@title Show final-pressure distribution and limit { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(technical_result.final_pressure_bar, bins=50, color="#9ecae1", edgecolor="white")
ax.axvline(pressure_endpoint_bar, color="#c00000", linestyle="--", linewidth=2, label=f"Pressure endpoint: {pressure_endpoint_bar:g} bar")
ax.set(xlabel="Screening final pressure (bar absolute)", ylabel="Iterations", title=technical_case.name)
ax.legend()
plt.show()

## Bibliographic references

- Nielsen, C.M. (2020). *Dynamic storage capacity evaluation for the Hanstholm and Havnsø structures*. GEUS Report 2020/48. [Official PDF](https://data.geus.dk/pure-pdf/GEUS-R_2020_48_web.pdf). Source for the Eclipse workflow, Figure 1 porosity-permeability equations, reference injection case, pressure constraint and sensitivity scenarios.
- Kristensen, L. (2020). *Reservoir data - Stenlille area*. GEUS Report 2020/28. [Official PDF](https://data.geus.dk/pure-pdf/GEUS-R_2020_28_web.pdf). Primary report for the Stenlille log/core database and porosity-permeability analogues extrapolated to Havnsø.
- Gregersen, U., Vosgerau, H., Smit, F.W.H., et al. (2023). *The Havnsø structure - Seismic data and interpretation to mature potential geological storage of CO₂*. GEUS Report 2023/38. [DOI](https://doi.org/10.22008/gpub/34705). Source for the updated Scenario 1 static-capacity inputs.

The normalized pressure and injectivity surrogate equations are original equations in this project. GEUS 2020/48 supplies their reference point and qualitative sensitivity directions, not the equations themselves.

## Interpretation and next calibration step

The combined success percentage is a **probabilistic screening result**, not yet a calibrated geological-risk probability. Capacity uncertainty comes from GEUS 2023/38. The permeability-factor probability distribution is derived from the 0.5× and 2× sensitivity scenarios in GEUS 2020/48; GEUS did not assign probabilities to them.

Before using the result for a project decision, replace this surrogate with an updated dynamic reservoir model based on the 2023 geometry and obtain numerical pressure responses, site-specific permeability/relative-permeability data, fracture-pressure measurements, and well-design constraints.